# 00 — Data Acquisition

Fetches the **raw** external inputs and caches them. Re-running is cheap: an
existing cache is loaded and the download is skipped (set `force_refresh: true`
in `config.yaml`, or `force_refresh=True` per call, to re-download).

**Raw only.** This stage does *not* compute returns, volatility, technical
indicators, `days_since_fomc`, or any lag. Those look-ahead-sensitive transforms
live in the feature layer (`01_features_target`) so every lag sits in one place.

Outputs (in `cache/`):
- `indices_raw.parquet` — OHLCV price panel, columns `(Field, Ticker)`
- `macro_raw.parquet` — FRED series (daily, forward-filled) + `*_pct_change`
- `fomc_calendar_2000_present.csv` — FOMC meeting/decision dates

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.config import load_config
from src.data import (
    load_or_fetch,
    download_price_panel,
    apply_vvix_gate,
    download_fred,
    scrape_fomc_calendar,
)

# Existing root config modules (ticker lists)
from index_ticker import SP500, US_INDEX_TICKERS, MACRO_TICKERS_POST_2006, INT_TICKERS
from fred_macro import FRED_MACRO_TICKERS

cfg = load_config(PROJECT_ROOT / "config.yaml")
print(f"window     : {cfg.start} -> {cfg.end}  (ref from {cfg.ref_start})")
print(f"cache dir  : {cfg.cache_dir}")
print(f"force_refresh: {cfg.force_refresh}")

window     : 2000-01-01 -> 2026-06-30  (ref from 1999-12-20)
cache dir  : C:\Users\jackshang\Desktop\Projects\sp500-prediction\cache
force_refresh: False


## 1 — Price panel (yfinance)

`primary` (target + US indices) share the NYSE calendar. `reference` (VIX/VVIX,
DXY, international indices) may trade on other calendars, so they are forward-filled
onto the primary calendar. VVIX is dropped at load time when `start` predates VVIX
history (`vvix_min_start`), keeping the cache itself universal.

In [2]:
primary = [SP500] + US_INDEX_TICKERS
reference = MACRO_TICKERS_POST_2006 + INT_TICKERS   # VIX, VVIX, DXY + international

panel = load_or_fetch(
    cfg.indices_path,
    lambda: download_price_panel(
        primary=primary,
        reference=reference,
        start=cfg.start,
        end=cfg.end,
        ref_start=cfg.ref_start,
    ),
    force_refresh=cfg.force_refresh,
)

# Deterministic load-time filter (see docstring): drop VVIX if start < 2007.
panel = apply_vvix_gate(panel, cfg.start, vvix_min_start=cfg.vvix_min_start)

panel.tail(3)

[cache] HIT   indices_raw.parquet  (skipping fetch)
[vvix] start 2000-01-01 < 2007-01-01: dropped 5 VVIX column(s)


Field             Close                                                   \
Ticker        000001.SS    DX-Y.NYB     ^BVSP          ^DJI        ^FCHI   
date                                                                       
2026-06-25  4120.280762  101.430000  171990.0  51920.621094  8431.610352   
2026-06-26  4027.264893  101.360001  173295.0  51876.109375  8384.870117   
2026-06-29  4073.902100  101.110001  173205.0  52182.738281  8367.330078   

Field                                                              \
Ticker             ^FTSE        ^GDAXI        ^GSPC       ^GSPTSE   
date                                                                
2026-06-25  10529.900391  24994.830078  7357.490234  34850.199219   
2026-06-26  10508.000000  24671.220703  7354.020020  34980.000000   
2026-06-29  10484.200195  24626.890625  7440.430176  34823.800781   

Field                     ...       Volume                            \
Ticker             ^IXIC  ...      ^GSPTSE        ^IXIC        ^N225   
date                      ...                                          
2026-06-25  25358.599609  ...  252573400.0   9870860000  151000000.0   
2026-06-26  25297.619141  ...  278542300.0  16594580000  156900000.0   
2026-06-29  25820.140625  ...  295233900.0   9776230000  182400000.0   

Field                                                                     \
Ticker             ^NDX        ^NYA        ^RUT         ^STI   ^STOXX50E   
date                                                                       
2026-06-25   9870860000  5565760000  5565760000  327665100.0  22517200.0   
2026-06-26  16594580000  9105600000  9105600000  289794800.0  19255900.0   
2026-06-29   9776230000  5794570000  5794570000  213020700.0  18393200.0   

Field                       
Ticker          ^TWII ^VIX  
date                        
2026-06-25  7621600.0  0.0  
2026-06-26  7695600.0  0.0  
2026-06-29  6052400.0  0.0  

[3 rows x 90 columns]

## 2 — Macro (FRED)

Set your FRED key in the environment variable named by `acquire.fred_api_key_env`
(default `FRED_API_KEY`), e.g. `export FRED_API_KEY=...`. Series come from the
existing `fred_macro.FRED_MACRO_TICKERS`. Daily-series lagging happens later, in the
feature layer.

In [3]:
api_key = cfg.fred_api_key
if not api_key:
    raise RuntimeError(
        f"FRED API key not found. Set the {cfg.fred_api_key_env!r} environment "
        f"variable before running this cell."
    )

macro = load_or_fetch(
    cfg.macro_path,
    lambda: download_fred(
        FRED_MACRO_TICKERS,
        start=cfg.ref_start,   # a little history before `start` for forward-fill
        end=cfg.end,
        api_key=api_key,
        pause=cfg.request_pause_sec,
    ),
    force_refresh=cfg.force_refresh,
)

macro.tail(3)

[cache] MISS  macro_raw.parquet  (not cached) -> fetching...
[fred] downloading DAAA ...


[fred] downloading DFF ...
[fred] downloading DCOILWTICO ...
[fred] downloading DGS2 ...
[fred] downloading DGS10 ...
[fred] downloading TOTBKCR ...
[fred] downloading CORESTICKM159SFRBATL ...
[fred] downloading UNRATE ...
[fred] downloading INDPRO ...
[fred] downloading UMCSENT ...
[fred] downloading IR14270 ...
[fred] downloading GDP ...
[cache] SAVE  macro_raw.parquet  (9,690 rows x 24 cols)


,DAAA,DAAA_pct_change,DFF,DFF_pct_change,DCOILWTICO,DCOILWTICO_pct_change,DGS2,DGS2_pct_change,DGS10,DGS10_pct_change,...,UNRATE,UNRATE_pct_change,INDPRO,INDPRO_pct_change,UMCSENT,UMCSENT_pct_change,IR14270,IR14270_pct_change,GDP,GDP_pct_change
date,,,,,,,,,,,,,,,,,,,,,
2026-06-28,5.50,0.001821,3.63,0.0,70.30,-0.032613,4.07,-0.004890,4.38,-0.004545,...,4.2,-0.023256,102.6395,0.000769,44.8,-0.100402,165.0,-0.025399,31865.721,0.014104
2026-06-29,5.49,-0.001818,3.63,0.0,71.87,0.022333,4.10,0.007371,4.38,0.000000,...,4.2,-0.023256,102.6395,0.000769,44.8,-0.100402,165.0,-0.025399,31865.721,0.014104
2026-06-30,5.55,0.010929,3.63,0.0,70.56,-0.018227,4.14,0.009756,4.44,0.013699,...,4.2,-0.023256,102.6395,0.000769,44.8,-0.100402,165.0,-0.025399,31865.721,0.014104


## 3 — FOMC calendar

Historical archive pages (2000–2020) plus the live calendar page (2021+). Cached as
CSV for easy inspection. `days_since_fomc` is derived later against the trading
calendar, in the feature layer.

In [4]:
fomc = load_or_fetch(
    cfg.fomc_path,
    lambda: scrape_fomc_calendar(2000, 2020, include_recent=True, pause=cfg.request_pause_sec),
    force_refresh=cfg.force_refresh,
    saver=lambda df, p: df.to_csv(p, index=False),
    loader=lambda p: pd.read_csv(p, parse_dates=["date"]),
)

fomc.tail(3)

[cache] HIT   fomc_calendar_2000_present.csv  (skipping fetch)


,date,is_fomc_day,is_fomc_press_conference
367,2026-04-29,1,0
368,2026-06-16,0,1
369,2026-06-17,1,0


## 4 — Sanity summary

In [5]:
tickers = sorted(set(panel.columns.get_level_values("Ticker")))

print("PANEL")
print(f"  shape   : {panel.shape}")
print(f"  dates   : {panel.index.min().date()} -> {panel.index.max().date()}")
print(f"  tickers : {tickers}")
print(f"  VVIX in : {'^VVIX' in tickers}  (start {cfg.start} vs min {cfg.vvix_min_start})")

print("\nMACRO")
print(f"  shape   : {macro.shape}")
print(f"  dates   : {macro.index.min().date()} -> {macro.index.max().date()}")
levels = [c for c in macro.columns if not str(c).endswith('_pct_change')]
print(f"  series  : {levels}")

print("\nFOMC")
print(f"  shape        : {fomc.shape}")
print(f"  dates        : {fomc['date'].min().date()} -> {fomc['date'].max().date()}")
print(f"  decision days: {int(fomc['is_fomc_day'].sum())}")

# Missingness on the modelled window (post-warmup NaNs worth eyeballing):
window = panel.loc[pd.Timestamp(cfg.start):]
na_frac = window.isna().mean().sort_values(ascending=False)
print("\nTop NaN fractions in panel (from start date):")
print(na_frac.head(8).to_string())

PANEL
  shape   : (6661, 90)
  dates   : 2000-01-03 -> 2026-06-29
  tickers : ['000001.SS', 'DX-Y.NYB', '^BVSP', '^DJI', '^FCHI', '^FTSE', '^GDAXI', '^GSPC', '^GSPTSE', '^IXIC', '^N225', '^NDX', '^NYA', '^RUT', '^STI', '^STOXX50E', '^TWII', '^VIX']
  VVIX in : False  (start 2000-01-01 vs min 2007-01-01)

MACRO
  shape   : (9690, 24)
  dates   : 1999-12-20 -> 2026-06-30
  series  : ['DAAA', 'DFF', 'DCOILWTICO', 'DGS2', 'DGS10', 'TOTBKCR', 'CORESTICKM159SFRBATL', 'UNRATE', 'INDPRO', 'UMCSENT', 'IR14270', 'GDP']

FOMC
  shape        : (370, 3)
  dates        : 2000-02-01 -> 2026-06-17
  decision days: 215

Top NaN fractions in panel (from start date):
Field   Ticker   
Close   ^STOXX50E    0.273082
High    ^STOXX50E    0.273082
Open    ^STOXX50E    0.273082
Volume  ^STOXX50E    0.273082
Low     ^STOXX50E    0.273082
Close   ^FTSE        0.000000
        ^BVSP        0.000000
        ^DJI         0.000000
